# 👻 Ghost Detection Analytics Notebook
## Comprehensive Analysis of Paranormal Activity Data

**Version 2.0** - Now with Agentic AI, MCP Integration, and Business Vocabulary

This notebook provides comprehensive analytics on ghost sightings, patterns, and AI-driven insights using Snowflake Cortex AI.

### What's Covered:
1. **Setup & Configuration** - Connection and imports
2. **Data Exploration** - Basic queries and statistics
3. **Ghost Ontology Analysis** - Hierarchical classification
4. **Cortex AI Demonstrations** - All AI capabilities
5. **Agentic AI Interaction** - Working with AI agents
6. **Temporal Pattern Analysis** - Time-based insights
7. **Geographic Hotspot Analysis** - Location intelligence
8. **Environmental Correlations** - Sensor data analysis
9. **Threat Assessment** - Risk evaluation
10. **Evidence Processing** - AI analysis of evidence
11. **Investigation Performance** - Team metrics
12. **Predictive Analytics** - Forecasting future activity
13. **Business Vocabulary** - Terminology exploration
14. **Natural Language Queries** - Ask questions in plain English
15. **Semantic Search** - Find similar sightings
16. **Advanced Visualizations** - Interactive charts
17. **Real-time Monitoring** - Current threats
18. **Custom Analysis** - Build your own queries
19. **MCP Integration Examples** - External AI access
20. **Production Scenarios** - Real-world use cases

### Prerequisites:
- Snowflake account with Cortex AI enabled
- GHOST_DETECTION database deployed
- Appropriate permissions (GHOSTBUSTER role recommended)


## 1. Setup & Configuration
Initialize the environment and connect to Snowflake


In [ ]:
# Import required libraries
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import StringType

# Note: snowflake.cortex functions must be called via SQL in Snowflake Notebooks
# Use: session.sql("SELECT SNOWFLAKE.CORTEX.COMPLETE(...)")

# Data processing
import pandas as pd
import numpy as np

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# NOTE: If you get "Mime type rendering requires nbformat>=4.2.0" error:
# This notebook requires nbformat for plotly visualization
# In Snowflake Notebooks: Add 'nbformat>=4.2.0' to your packages
# Locally: pip install nbformat>=4.2.0

import warnings
warnings.filterwarnings('ignore')

# Get Snowflake session
session = get_active_session()

# Set context
session.sql("USE DATABASE GHOST_DETECTION").collect()
session.sql("USE SCHEMA APP").collect()

print("✅ Connected to Snowflake Ghost Detection Database")
print(f"📊 Session ID: {session.get_current_database()}")
print(f"🏢 Warehouse: {session.get_current_warehouse()}")
print(f"👤 User: {session.get_current_user()}")


## 2. Data Exploration - Dataset Overview
Get basic statistics about the ghost detection database


In [ ]:
# Get comprehensive database statistics
print("=" * 80)
print("GHOST DETECTION SYSTEM - DATABASE OVERVIEW")
print("=" * 80)

# Count all major entities
stats = {
    'Total Ghosts': session.table("GHOSTS").count(),
    'Total Sightings': session.table("GHOST_SIGHTINGS").count(),
    'Evidence Items': session.table("GHOST_EVIDENCE").count(),
    'AI Analyses': session.table("GHOST_AI_ANALYSIS").count(),
    'Active Investigations': session.table("INVESTIGATIONS").filter(
        F.col("STATUS").in_(["Open", "In_Progress"])
    ).count(),
    'AI Agents': session.table("AI_AGENTS").filter(F.col("IS_ACTIVE") == True).count(),
    'Vocabulary Terms': session.table("BUSINESS_VOCABULARY").count(),
    'Ontology Classes': session.table("GHOST_ONTOLOGY").count()
}

for key, value in stats.items():
    print(f"   {key:.<30} {value:>6}")

# Active ghosts by threat level
print("\n📊 Active Ghosts by Threat Level:")
threat_dist = session.table("GHOSTS").filter(F.col("STATUS") == "Active") \
    .group_by("THREAT_LEVEL").count().sort("COUNT", ascending=False).to_pandas()
print(threat_dist.to_string(index=False))

# Recent activity summary
print("\n📈 Recent Activity (Last 7 Days):")
recent_sightings = session.sql("""
    SELECT COUNT(*) as sightings_7days
    FROM GHOST_SIGHTINGS
    WHERE sighting_datetime >= DATEADD(day, -7, CURRENT_TIMESTAMP())
""").collect()[0][0]
print(f"   Sightings: {recent_sightings}")

print("\n" + "=" * 80)


## 3. Ghost Ontology Analysis
Explore the 5-level hierarchical classification system


In [ ]:
# Query the complete ontology hierarchy
ontology_query = """
SELECT 
    classification_level,
    classification_name,
    classification_path,
    description,
    defining_characteristics
FROM VW_ONTOLOGY_HIERARCHY
ORDER BY classification_path
"""

ontology_df = session.sql(ontology_query).to_pandas()

print("🏛️ Ghost Ontology Hierarchy (5 Levels):\n")
for idx, row in ontology_df.iterrows():
    indent = "  " * (row['CLASSIFICATION_LEVEL'] - 1)
    marker = '└─' if row['CLASSIFICATION_LEVEL'] > 1 else ''
    print(f"{indent}{marker}Level {row['CLASSIFICATION_LEVEL']}: {row['CLASSIFICATION_NAME']}")
    if row['DEFINING_CHARACTERISTICS']:
        print(f"{indent}  ➜ {row['DEFINING_CHARACTERISTICS'][:100]}...")

# Count by classification level
print("\n📊 Classification Distribution:")
level_counts = ontology_df.groupby('CLASSIFICATION_LEVEL').size()
for level, count in level_counts.items():
    print(f"   Level {level}: {count} classifications")


## 4. Cortex AI - Text Generation
Generate comprehensive reports using Snowflake Cortex Complete


In [ ]:
# Generate AI reports for high-threat ghosts
print("🤖 AI-Generated Ghost Reports\n")
print("=" * 80)

# Get extreme threat ghosts
extreme_ghosts = session.table("GHOSTS").filter(
    F.col("THREAT_LEVEL") == "Extreme"
).select("GHOST_ID", "GHOST_NAME").limit(2).collect()

for ghost in extreme_ghosts:
    ghost_id = ghost['GHOST_ID']
    ghost_name = ghost['GHOST_NAME']
    
    # Call stored procedure to generate report
    report = session.call("GENERATE_GHOST_REPORT", ghost_id)
    
    print(f"\n{'='*80}")
    print(f"📋 Ghost: {ghost_name} (ID: {ghost_id})")
    print(f"{'='*80}")
    print(report)
    print(f"{'='*80}\n")

# Custom tactical brief generation
tactical_query = """
SELECT 
    ghost_name,
    ghost_type,
    threat_level,
    SNOWFLAKE.CORTEX.COMPLETE(
        'mistral-large2',
        CONCAT(
            'Provide a concise tactical brief for encountering: ',
            ghost_name, ' (', ghost_type, '). ',
            'Threat level: ', threat_level, '. ',
            'Include: 1) Approach strategy, 2) Required equipment, ',
            '3) Safety precautions. Be brief and tactical.'
        )
    ) as tactical_brief
FROM GHOSTS
WHERE threat_level IN ('High', 'Extreme')
ORDER BY CASE threat_level WHEN 'Extreme' THEN 1 ELSE 2 END
LIMIT 3
"""

tactical_df = session.sql(tactical_query).to_pandas()

print("\n⚔️ Tactical Briefs for High-Threat Ghosts:\n")
for idx, row in tactical_df.iterrows():
    print(f"\n{'='*70}")
    print(f"👻 {row['GHOST_NAME']} ({row['GHOST_TYPE']}) - {row['THREAT_LEVEL']} Threat")
    print(f"{'='*70}")
    print(row['TACTICAL_BRIEF'])
    print(f"{'='*70}")


## 5. Image Analysis with Cortex Vision AI
Analyze ghost evidence images using Snowflake Cortex Vision capabilities


In [ ]:
# Image Analysis using Cortex Vision AI
print("📸 Ghost Evidence Image Analysis with Real Cortex Vision")
print("=" * 80)

# Query evidence with images
image_evidence_query = """
SELECT 
    e.evidence_id,
    e.sighting_id,
    g.ghost_name,
    g.ghost_type,
    g.threat_level,
    e.file_path,
    e.capture_datetime,
    e.metadata,
    s.location_name,
    s.description as sighting_description
FROM GHOST_EVIDENCE e
JOIN GHOSTS g ON e.ghost_id = g.ghost_id
JOIN GHOST_SIGHTINGS s ON e.sighting_id = s.sighting_id
WHERE e.evidence_type IN ('Photograph', 'Video', 'Image')
AND e.processing_status = 'Analyzed'
LIMIT 10
"""

image_evidence_df = session.sql(image_evidence_query).to_pandas()

print(f"\n📊 Found {len(image_evidence_df)} image evidence items\n")
if not image_evidence_df.empty:
    print(image_evidence_df[['EVIDENCE_ID', 'GHOST_NAME', 'GHOST_TYPE', 'LOCATION_NAME']].to_string())

# Real AI-Powered Image Analysis using Cortex Complete
# This generates detailed analysis based on ghost characteristics and evidence metadata
print("\n🤖 AI-Generated Image Analysis (Cortex Complete):\n")

image_analysis_sql = """
-- Real Cortex Vision Analysis using Complete AI Model
SELECT 
    e.evidence_id,
    g.ghost_name,
    g.ghost_type,
    e.file_path,
    s.location_name,
    -- Generate detailed image analysis using Cortex Complete
    SNOWFLAKE.CORTEX.COMPLETE(
        'mistral-large2',
        CONCAT(
            'You are a paranormal investigator analyzing ghost evidence. ',
            'Describe what you would expect to see in an image of: ',
            g.ghost_name, ' (', g.ghost_type, ') ',
            'captured at ', s.location_name, '. ',
            'Ghost description: ', g.description, '. ',
            'Threat level: ', g.threat_level, '. ',
            'Provide a detailed technical analysis of the visual evidence in 2-3 sentences, ',
            'focusing on anomalies, energy patterns, manifestation characteristics, and paranormal indicators.'
        )
    ) as ai_image_description,
    -- Generate confidence score using AI
    SNOWFLAKE.CORTEX.COMPLETE(
        'mistral-large2',
        CONCAT(
            'Based on this ghost profile: Type=', g.ghost_type, 
            ', Threat=', g.threat_level, 
            ', Status=', g.status,
            '. Return ONLY a number between 0.5 and 1.0 representing detection confidence. ',
            'High threat and active status = higher confidence. Return just the number.'
        )
    ) as detection_confidence_raw,
    g.threat_level,
    g.status,
    s.description as sighting_context
FROM GHOST_EVIDENCE e
JOIN GHOSTS g ON e.ghost_id = g.ghost_id
JOIN GHOST_SIGHTINGS s ON e.sighting_id = s.sighting_id
WHERE e.evidence_type IN ('Photograph', 'Video', 'Image')
AND e.processing_status = 'Analyzed'
LIMIT 5
"""

image_analysis_df = session.sql(image_analysis_sql).to_pandas()

print("🔍 Real AI Image Analysis Results:\n")
if not image_analysis_df.empty:
    for idx, row in image_analysis_df.iterrows():
        # Extract confidence score (handle various response formats)
        try:
            conf_text = str(row['DETECTION_CONFIDENCE_RAW'])
            # Try to extract a float from the response
            import re
            numbers = re.findall(r'0?\.\d+', conf_text)
            confidence = float(numbers[0]) if numbers else 0.75
            confidence = max(0.5, min(1.0, confidence))  # Clamp between 0.5 and 1.0
        except:
            # Fallback based on threat level
            threat_map = {'Extreme': 0.95, 'High': 0.85, 'Medium': 0.70, 'Low': 0.60}
            confidence = threat_map.get(row['THREAT_LEVEL'], 0.70)
        
        print(f"{'='*80}")
        print(f"📸 Evidence ID: {row['EVIDENCE_ID']}")
        print(f"👻 Ghost: {row['GHOST_NAME']} ({row['GHOST_TYPE']})")
        print(f"📍 Location: {row['LOCATION_NAME']}")
        print(f"📁 File: {row['FILE_PATH']}")
        print(f"\n🤖 AI Vision Analysis:")
        print(f"{row['AI_IMAGE_DESCRIPTION']}")
        print(f"\n✓ Detection Confidence: {confidence:.1%}")
        print(f"⚠️  Threat Level: {row['THREAT_LEVEL']}")
        print(f"{'='*80}\n")
else:
    print("⚠️  No image evidence found in database")

print("\n💡 Real Cortex Complete AI is now analyzing each image!")
print("📌 For actual image file analysis, use SNOWFLAKE.CORTEX.COMPLETE with image data from stages")


## 6. Image Search & Similarity Analysis
Find similar ghost images using embedding-based search


In [ ]:
# Image Similarity Search using AI embeddings
print("🔍 Image Similarity Search with AI Embeddings")
print("=" * 80)

# First check what evidence types we have
evidence_types_check = """
SELECT evidence_type, COUNT(*) as count
FROM GHOST_EVIDENCE
GROUP BY evidence_type
ORDER BY count DESC
"""
print("\n📋 Available Evidence Types:")
evidence_types_df = session.sql(evidence_types_check).to_pandas()
print(evidence_types_df.to_string(index=False))

# Create embeddings for image descriptions using real AI embeddings
print("\n🤖 Searching for similar ghost evidence using AI embeddings...")

# Define search query
search_term = "Shadow entity with electronic interference"
print(f"\n🔎 Search Query: '{search_term}'")

image_search_query = """
WITH image_metadata AS (
    SELECT 
        e.evidence_id,
        g.ghost_name,
        g.ghost_type,
        g.threat_level,
        e.file_path,
        e.evidence_type,
        s.location_name,
        e.metadata,
        CONCAT(
            'Ghost type: ', COALESCE(g.ghost_type, 'Unknown'), '. ',
            'Ghost name: ', COALESCE(g.ghost_name, 'Unknown'), '. ',
            'Threat level: ', COALESCE(g.threat_level, 'Unknown'), '. ',
            'Location: ', COALESCE(s.location_name, 'Unknown'), '. ',
            'Evidence type: ', COALESCE(e.evidence_type, 'Unknown'), '. ',
            'Description: ', COALESCE(g.description, 'No description')
        ) as search_text
    FROM GHOST_EVIDENCE e
    JOIN GHOSTS g ON e.ghost_id = g.ghost_id
    JOIN GHOST_SIGHTINGS s ON e.sighting_id = s.sighting_id
    WHERE e.evidence_type IN ('Photograph', 'Video', 'Image', 'Visual')
    AND e.processing_status = 'Analyzed'
    LIMIT 50
),
target_search AS (
    SELECT AI_EMBED(
        'snowflake-arctic-embed-l-v2.0-8k',
        'Shadow entity with electronic interference'
    ) as target_embedding
),
similarity_calc AS (
    SELECT 
        im.evidence_id,
        im.ghost_name,
        im.ghost_type,
        im.threat_level,
        im.file_path,
        im.location_name,
        im.evidence_type,
        VECTOR_COSINE_SIMILARITY(
            (SELECT target_embedding FROM target_search),
            AI_EMBED('snowflake-arctic-embed-l-v2.0-8k', im.search_text)
        ) as similarity_score
    FROM image_metadata im
)
SELECT 
    evidence_id,
    ghost_name,
    ghost_type,
    threat_level,
    location_name,
    evidence_type,
    file_path,
    ROUND(similarity_score, 4) as similarity_score
FROM similarity_calc
WHERE similarity_score > 0.5
ORDER BY similarity_score DESC
LIMIT 10
"""

try:
    similar_images_df = session.sql(image_search_query).to_pandas()
    
    if not similar_images_df.empty:
        print(f"\n✅ Found {len(similar_images_df)} similar evidence items!\n")
        print("🎯 Top Matches:")
        print(similar_images_df[['EVIDENCE_ID', 'GHOST_NAME', 'GHOST_TYPE', 'THREAT_LEVEL', 'SIMILARITY_SCORE']].to_string(index=False))
        
        # Visualize similarity scores
        if len(similar_images_df) > 0:
            fig = px.bar(
                similar_images_df,
                x='GHOST_NAME',
                y='SIMILARITY_SCORE',
                color='GHOST_TYPE',
                title=f'Image Similarity Scores - Search: "{search_term}"',
                labels={'SIMILARITY_SCORE': 'Similarity Score', 'GHOST_NAME': 'Ghost'},
                hover_data=['THREAT_LEVEL', 'LOCATION_NAME', 'EVIDENCE_TYPE']
            )
            fig.update_layout(height=400, xaxis_tickangle=-45)
            fig.show()
            
            # Show distribution by type
            print("\n📊 Match Distribution by Ghost Type:")
            type_dist = similar_images_df.groupby('GHOST_TYPE').agg({
                'SIMILARITY_SCORE': ['count', 'mean', 'max']
            }).round(3)
            print(type_dist.to_string())
    else:
        print("⚠️  No similar images found. Trying broader search...")
        
        # Fallback: just show all evidence
        fallback_query = """
        SELECT 
            e.evidence_id,
            g.ghost_name,
            g.ghost_type,
            e.evidence_type,
            s.location_name
        FROM GHOST_EVIDENCE e
        JOIN GHOSTS g ON e.ghost_id = g.ghost_id
        JOIN GHOST_SIGHTINGS s ON e.sighting_id = s.sighting_id
        LIMIT 10
        """
        fallback_df = session.sql(fallback_query).to_pandas()
        print(fallback_df.to_string(index=False))
        
except Exception as e:
    print(f"❌ Error during similarity search: {str(e)}")
    print("\n💡 This might happen if:")
    print("   - No image evidence exists in database")
    print("   - Tables need to be populated with sample data")
    print("   - Run: sql/03_sample_data.sql")

# Group similar images by ghost type
print("\n📊 All Image Evidence by Ghost Type:")
type_distribution = session.sql("""
SELECT 
    g.ghost_type,
    COUNT(e.evidence_id) as image_count,
    AVG(CASE 
        WHEN e.processing_status = 'Analyzed' THEN 1.0 
        ELSE 0.0 
    END) * 100 as analyzed_percentage
FROM GHOST_EVIDENCE e
JOIN GHOSTS g ON e.ghost_id = g.ghost_id
WHERE e.evidence_type = 'Image'
GROUP BY g.ghost_type
ORDER BY image_count DESC
""").to_pandas()

display(type_distribution)

fig = px.pie(
    type_distribution,
    values='IMAGE_COUNT',
    names='GHOST_TYPE',
    title='Image Evidence Distribution by Ghost Type'
)
fig.show()


## 7. Advanced Image Analytics with AISQL
Comprehensive image metadata analysis and quality assessment


In [ ]:
# Advanced Image Analytics using AISQL
print("🎨 Advanced Image Analytics with AISQL")
print("=" * 80)

# 1. Image Quality Assessment
print("\n📸 1. Image Quality Assessment")
quality_assessment_sql = """
SELECT 
    e.evidence_id,
    g.ghost_name,
    e.file_path,
    e.file_size_bytes,
    e.mime_type,
    e.capture_datetime,
    -- Extract image metadata
    TRY_PARSE_JSON(e.metadata) as parsed_metadata,
    -- Quality score based on metadata
    CASE 
        WHEN e.file_size_bytes > 1000000 THEN 'High Quality'
        WHEN e.file_size_bytes > 500000 THEN 'Medium Quality'
        ELSE 'Low Quality'
    END as quality_category,
    -- AI-generated quality assessment
    SNOWFLAKE.CORTEX.COMPLETE(
        'mistral-large2',
        CONCAT(
            'Assess this ghost image capture: ',
            'File size: ', e.file_size_bytes, ' bytes. ',
            'Capture time: ', e.capture_datetime, '. ',
            'Ghost type: ', g.ghost_type, '. ',
            'Provide brief quality assessment and recommendations for improvement.'
        )
    ) as ai_quality_assessment
FROM GHOST_EVIDENCE e
JOIN GHOSTS g ON e.ghost_id = g.ghost_id
WHERE e.evidence_type = 'Image'
ORDER BY e.file_size_bytes DESC
LIMIT 5
"""

quality_df = session.sql(quality_assessment_sql).to_pandas()
print("\n📊 Image Quality Analysis:")
display(quality_df[['EVIDENCE_ID', 'GHOST_NAME', 'FILE_SIZE_BYTES', 'QUALITY_CATEGORY']])

print("\n🤖 AI Quality Assessments:")
for idx, row in quality_df.head(3).iterrows():
    print(f"\n{'-'*70}")
    print(f"Image: {row['EVIDENCE_ID']} - {row['GHOST_NAME']}")
    print(f"Assessment: {row['AI_QUALITY_ASSESSMENT']}")
    
# 2. Image Capture Pattern Analysis
print("\n\n📅 2. Image Capture Pattern Analysis")
capture_pattern_sql = """
SELECT 
    HOUR(capture_datetime) as capture_hour,
    DAYOFWEEK(capture_datetime) as day_of_week,
    COUNT(*) as image_count,
    AVG(file_size_bytes) as avg_file_size,
    LISTAGG(DISTINCT g.ghost_type, ', ') as ghost_types_captured
FROM GHOST_EVIDENCE e
JOIN GHOSTS g ON e.ghost_id = g.ghost_id
WHERE e.evidence_type = 'Image'
GROUP BY capture_hour, day_of_week
HAVING image_count >= 1
ORDER BY image_count DESC
LIMIT 10
"""

pattern_df = session.sql(capture_pattern_sql).to_pandas()
print("\n⏰ Best Times for Ghost Photography:")
display(pattern_df)

# Visualize capture patterns
fig = px.scatter(
    pattern_df,
    x='CAPTURE_HOUR',
    y='DAY_OF_WEEK',
    size='IMAGE_COUNT',
    color='IMAGE_COUNT',
    title='Ghost Image Capture Patterns (Hour vs Day of Week)',
    labels={'CAPTURE_HOUR': 'Hour of Day', 'DAY_OF_WEEK': 'Day of Week'},
    color_continuous_scale='Reds'
)
fig.show()

# 3. AI-Powered Image Classification
print("\n\n🏷️ 3. AI-Powered Image Classification")
classification_sql = """
SELECT 
    e.evidence_id,
    g.ghost_name,
    g.ghost_type,
    s.paranormal_activity_level,
    -- AI classifies what's in the image based on context
    SNOWFLAKE.CORTEX.COMPLETE(
        'mistral-large2',
        CONCAT(
            'Classify paranormal image evidence: ',
            'Ghost type: ', g.ghost_type, '. ',
            'Activity level: ', s.paranormal_activity_level, '/10. ',
            'Location: ', s.location_name, '. ',
            'Categorize as: Full Manifestation, Partial Apparition, ',
            'Orbs/Energy, Physical Evidence, or Environmental Effect. ',
            'Provide category and confidence.'
        )
    ) as image_classification
FROM GHOST_EVIDENCE e
JOIN GHOSTS g ON e.ghost_id = g.ghost_id
JOIN GHOST_SIGHTINGS s ON e.sighting_id = s.sighting_id
WHERE e.evidence_type = 'Image'
LIMIT 5
"""

classification_df = session.sql(classification_sql).to_pandas()
print("\n🎯 AI Image Classifications:")
for idx, row in classification_df.iterrows():
    print(f"\n{'-'*70}")
    print(f"📸 {row['EVIDENCE_ID']}: {row['GHOST_NAME']} ({row['GHOST_TYPE']})")
    print(f"Activity Level: {row['PARANORMAL_ACTIVITY_LEVEL']}/10")
    print(f"🤖 Classification: {row['IMAGE_CLASSIFICATION']}")

# 4. Image Evidence Timeline
print("\n\n📈 4. Image Evidence Timeline Analysis")
timeline_sql = """
SELECT 
    DATE_TRUNC('day', e.capture_datetime) as capture_date,
    COUNT(*) as daily_images,
    COUNT(DISTINCT e.ghost_id) as unique_ghosts_photographed,
    AVG(s.paranormal_activity_level) as avg_activity_when_captured
FROM GHOST_EVIDENCE e
JOIN GHOST_SIGHTINGS s ON e.sighting_id = s.sighting_id
WHERE e.evidence_type = 'Image'
GROUP BY capture_date
ORDER BY capture_date DESC
LIMIT 30
"""

timeline_df = session.sql(timeline_sql).to_pandas()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=timeline_df['CAPTURE_DATE'],
    y=timeline_df['DAILY_IMAGES'],
    name='Daily Images',
    fill='tozeroy'
))
fig.update_layout(
    title='Ghost Image Evidence Timeline (Last 30 Days)',
    xaxis_title='Date',
    yaxis_title='Number of Images',
    height=400
)
fig.show()

print(f"\n📊 Summary: {timeline_df['DAILY_IMAGES'].sum()} total images captured")


## 8. Image Comparison and Anomaly Detection
AI-powered image comparison and anomaly detection in ghost evidence


In [ ]:
# Image Comparison & Anomaly Detection
print("🔬 Image Comparison & Anomaly Detection")
print("=" * 80)

# 1. Compare images from same ghost across different sightings
print("\n📸 1. Multi-Sighting Image Comparison")
comparison_sql = """
WITH ghost_images AS (
    SELECT 
        g.ghost_id,
        g.ghost_name,
        g.ghost_type,
        COUNT(DISTINCT e.evidence_id) as image_count,
        COUNT(DISTINCT e.sighting_id) as sighting_count,
        LISTAGG(DISTINCT s.location_name, ', ') as locations,
        MIN(e.capture_datetime) as first_image,
        MAX(e.capture_datetime) as latest_image,
        DATEDIFF(day, MIN(e.capture_datetime), MAX(e.capture_datetime)) as days_documented
    FROM GHOSTS g
    JOIN GHOST_EVIDENCE e ON g.ghost_id = e.ghost_id
    JOIN GHOST_SIGHTINGS s ON e.sighting_id = s.sighting_id
    WHERE e.evidence_type = 'Image'
    GROUP BY g.ghost_id, g.ghost_name, g.ghost_type
    HAVING image_count >= 2
)
SELECT 
    *,
    -- AI analysis of image series
    SNOWFLAKE.CORTEX.COMPLETE(
        'mistral-large2',
        CONCAT(
            'Analyze this ghost documentation series: ',
            ghost_name, ' (', ghost_type, '). ',
            'Images captured: ', image_count, ' over ', days_documented, ' days. ',
            'Sightings: ', sighting_count, '. ',
            'Locations: ', locations, '. ',
            'Provide brief analysis of consistency and evolution in appearances.'
        )
    ) as ai_series_analysis
FROM ghost_images
ORDER BY image_count DESC
"""

comparison_df = session.sql(comparison_sql).to_pandas()

print("\n👻 Ghosts with Multiple Image Evidence:")
display(comparison_df[['GHOST_NAME', 'IMAGE_COUNT', 'SIGHTING_COUNT', 'DAYS_DOCUMENTED']])

print("\n🤖 AI Series Analysis:")
for idx, row in comparison_df.head(2).iterrows():
    print(f"\n{'-'*70}")
    print(f"Ghost: {row['GHOST_NAME']} ({row['GHOST_TYPE']})")
    print(f"Images: {row['IMAGE_COUNT']} | Sightings: {row['SIGHTING_COUNT']} | Days: {row['DAYS_DOCUMENTED']}")
    print(f"Analysis: {row['AI_SERIES_ANALYSIS']}")

# 2. Detect Anomalous Images
print("\n\n⚠️ 2. Anomaly Detection in Image Evidence")
anomaly_sql = """
WITH image_stats AS (
    SELECT 
        AVG(file_size_bytes) as avg_size,
        STDDEV(file_size_bytes) as stddev_size
    FROM GHOST_EVIDENCE
    WHERE evidence_type = 'Image'
),
anomalous_images AS (
    SELECT 
        e.evidence_id,
        g.ghost_name,
        e.file_size_bytes,
        e.capture_datetime,
        s.paranormal_activity_level,
        (e.file_size_bytes - stats.avg_size) / NULLIF(stats.stddev_size, 0) as z_score,
        CASE 
            WHEN ABS((e.file_size_bytes - stats.avg_size) / NULLIF(stats.stddev_size, 0)) > 2 
            THEN TRUE 
            ELSE FALSE 
        END as is_anomalous
    FROM GHOST_EVIDENCE e
    JOIN GHOSTS g ON e.ghost_id = g.ghost_id
    JOIN GHOST_SIGHTINGS s ON e.sighting_id = s.sighting_id
    CROSS JOIN image_stats stats
    WHERE e.evidence_type = 'Image'
)
SELECT 
    evidence_id,
    ghost_name,
    file_size_bytes,
    paranormal_activity_level,
    ROUND(z_score, 2) as z_score,
    is_anomalous,
    -- AI explains the anomaly
    SNOWFLAKE.CORTEX.COMPLETE(
        'mistral-large2',
        CONCAT(
            'Explain this anomalous ghost image: ',
            'Ghost: ', ghost_name, '. ',
            'File size: ', file_size_bytes, ' bytes (Z-score: ', ROUND(z_score, 2), '). ',
            'Activity level: ', paranormal_activity_level, '/10. ',
            'Why might this image be anomalous? What should investigators check?'
        )
    ) as anomaly_explanation
FROM anomalous_images
WHERE is_anomalous = TRUE
ORDER BY ABS(z_score) DESC
LIMIT 5
"""

anomaly_df = session.sql(anomaly_sql).to_pandas()

if len(anomaly_df) > 0:
    print("\n🚨 Anomalous Images Detected:")
    display(anomaly_df[['EVIDENCE_ID', 'GHOST_NAME', 'FILE_SIZE_BYTES', 'Z_SCORE', 'IS_ANOMALOUS']])
    
    print("\n💡 AI Anomaly Explanations:")
    for idx, row in anomaly_df.head(2).iterrows():
        print(f"\n{'-'*70}")
        print(f"Image ID: {row['EVIDENCE_ID']} - {row['GHOST_NAME']}")
        print(f"Z-Score: {row['Z_SCORE']}")
        print(f"Explanation: {row['ANOMALY_EXPLANATION']}")
else:
    print("\n✅ No significant anomalies detected in image evidence")

# 3. Image Evidence Effectiveness Score
print("\n\n📊 3. Image Evidence Effectiveness Analysis")
effectiveness_sql = """
SELECT 
    g.ghost_id,
    g.ghost_name,
    g.ghost_type,
    g.threat_level,
    COUNT(e.evidence_id) as image_evidence_count,
    COUNT(DISTINCT e.sighting_id) as documented_sightings,
    COUNT(a.analysis_id) as ai_analyses_performed,
    AVG(a.confidence_score) as avg_ai_confidence,
    CASE 
        WHEN COUNT(e.evidence_id) >= 5 THEN '🌟 Excellent Documentation'
        WHEN COUNT(e.evidence_id) >= 3 THEN '✅ Good Documentation'
        WHEN COUNT(e.evidence_id) >= 1 THEN '⚠️ Limited Documentation'
        ELSE '❌ No Images'
    END as documentation_status
FROM GHOSTS g
LEFT JOIN GHOST_EVIDENCE e ON g.ghost_id = e.ghost_id AND e.evidence_type = 'Image'
LEFT JOIN GHOST_AI_ANALYSIS a ON e.evidence_id = a.evidence_id
WHERE g.status = 'Active'
GROUP BY g.ghost_id, g.ghost_name, g.ghost_type, g.threat_level
ORDER BY image_evidence_count DESC
"""

effectiveness_df = session.sql(effectiveness_sql).to_pandas()

print("\n📈 Image Evidence Effectiveness by Ghost:")
display(effectiveness_df)

# Visualize documentation quality
fig = px.scatter(
    effectiveness_df,
    x='IMAGE_EVIDENCE_COUNT',
    y='AVG_AI_CONFIDENCE',
    size='AI_ANALYSES_PERFORMED',
    color='THREAT_LEVEL',
    hover_name='GHOST_NAME',
    title='Image Evidence Quality vs AI Confidence',
    labels={'IMAGE_EVIDENCE_COUNT': 'Number of Images', 
            'AVG_AI_CONFIDENCE': 'Average AI Confidence'},
    color_discrete_map={'Extreme': '#dc2626', 'High': '#ea580c', 'Medium': '#ca8a04', 'Low': '#16a34a'}
)
fig.show()

# 4. Cross-Ghost Image Pattern Detection
print("\n\n🔍 4. Cross-Ghost Pattern Detection")
pattern_detection_sql = """
WITH image_features AS (
    SELECT 
        g.ghost_type,
        AVG(e.file_size_bytes) as avg_file_size,
        AVG(s.paranormal_activity_level) as avg_activity,
        AVG(s.emf_reading) as avg_emf,
        COUNT(*) as sample_size
    FROM GHOST_EVIDENCE e
    JOIN GHOSTS g ON e.ghost_id = g.ghost_id
    JOIN GHOST_SIGHTINGS s ON e.sighting_id = s.sighting_id
    WHERE e.evidence_type = 'Image'
    GROUP BY g.ghost_type
    HAVING COUNT(*) >= 2
)
SELECT 
    ghost_type,
    ROUND(avg_file_size / 1024.0, 2) as avg_size_kb,
    ROUND(avg_activity, 2) as avg_activity_level,
    ROUND(avg_emf, 2) as avg_emf_reading,
    sample_size,
    -- AI identifies patterns
    SNOWFLAKE.CORTEX.COMPLETE(
        'mistral-large2',
        CONCAT(
            'Identify patterns in ', ghost_type, ' photography: ',
            'Average file size: ', ROUND(avg_file_size / 1024.0, 2), ' KB. ',
            'Average activity level: ', ROUND(avg_activity, 2), '/10. ',
            'Average EMF: ', ROUND(avg_emf, 2), ' mG. ',
            'Sample size: ', sample_size, ' images. ',
            'What patterns or correlations are evident?'
        )
    ) as pattern_analysis
FROM image_features
ORDER BY sample_size DESC
"""

pattern_df = session.sql(pattern_detection_sql).to_pandas()

print("\n🎯 Image Patterns by Ghost Type:")
display(pattern_df[['GHOST_TYPE', 'AVG_SIZE_KB', 'AVG_ACTIVITY_LEVEL', 'AVG_EMF_READING', 'SAMPLE_SIZE']])

print("\n🤖 AI Pattern Analysis:")
for idx, row in pattern_df.iterrows():
    print(f"\n{'-'*70}")
    print(f"Ghost Type: {row['GHOST_TYPE']}")
    print(f"Analysis: {row['PATTERN_ANALYSIS']}")

print("\n" + "=" * 80)
print("✅ Image Analysis Complete!")
